In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output, FileLink

# ==========================================
# 1. GENERATE SAMPLE SALES DATA
# ==========================================

np.random.seed(42)

regions = ["North", "South", "East", "West"]
categories = ["Electronics", "Clothing", "Furniture", "Groceries"]

products = {
    "Electronics": ["Laptop", "Headphones", "Mouse", "Keyboard"],
    "Clothing": ["Shirt", "Jeans", "Jacket", "Shoes"],
    "Furniture": ["Chair", "Table", "Desk", "Bookshelf"],
    "Groceries": ["Rice", "Milk", "Bread", "Coffee"]
}

n = 200

dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2026-01-01", "2026-09-30"),
        n
    )
)

category_data = np.random.choice(categories, n)

data = pd.DataFrame({
    "Order_ID": [f"ORD-{i:04d}" for i in range(1, n + 1)],
    "Date": dates,
    "Region": np.random.choice(regions, n),
    "Category": category_data,
    "Product": [
        np.random.choice(products[c]) for c in category_data
    ],
    "Units": np.random.randint(1, 11, n),
    "Unit_Price": np.random.randint(100, 5001, n)
})

data["Revenue"] = data["Units"] * data["Unit_Price"]

data["Cost"] = (
    data["Revenue"] * np.random.uniform(0.50, 0.85, n)
).round(2)

data["Profit"] = (data["Revenue"] - data["Cost"]).round(2)

print("Sample dataset created successfully!")
print("Total records:", len(data))
display(data.head())

# ==========================================
# 2. DASHBOARD FILTERS
# ==========================================

region_filter = widgets.Dropdown(
    options=["All Regions"] + regions,
    value="All Regions",
    description="Region:"
)

category_filter = widgets.Dropdown(
    options=["All Categories"] + categories,
    value="All Categories",
    description="Category:"
)

output = widgets.Output()

def currency(value):
    return f"₹{value:,.2f}"

# ==========================================
# 3. DASHBOARD FUNCTION
# ==========================================

def update_dashboard(change=None):

    filtered = data.copy()

    if region_filter.value != "All Regions":
        filtered = filtered[
            filtered["Region"] == region_filter.value
        ]

    if category_filter.value != "All Categories":
        filtered = filtered[
            filtered["Category"] == category_filter.value
        ]

    with output:
        clear_output(wait=True)

        if filtered.empty:
            print("No data available for these filters.")
            return

        # Key Performance Indicators
        revenue = filtered["Revenue"].sum()
        profit = filtered["Profit"].sum()
        orders = filtered["Order_ID"].nunique()
        units = filtered["Units"].sum()
        average_order = revenue / orders if orders else 0
        profit_margin = profit / revenue * 100 if revenue else 0

        print("=" * 60)
        print("           SALES ANALYTICS DASHBOARD")
        print("=" * 60)

        print(f"Total Revenue      : {currency(revenue)}")
        print(f"Total Profit       : {currency(profit)}")
        print(f"Total Orders       : {orders:,}")
        print(f"Total Units Sold   : {units:,}")
        print(f"Average Order Value: {currency(average_order)}")
        print(f"Profit Margin      : {profit_margin:.2f}%")

        # ==================================
        # 4. PREPARE GRAPH DATA
        # ==================================

        monthly_revenue = (
            filtered.set_index("Date")["Revenue"]
            .resample("MS").sum()
        )

        monthly_profit = (
            filtered.set_index("Date")["Profit"]
            .resample("MS").sum()
        )

        category_revenue = (
            filtered.groupby("Category")["Revenue"]
            .sum().sort_values(ascending=False)
        )

        region_revenue = (
            filtered.groupby("Region")["Revenue"]
            .sum().sort_values()
        )

        # ==================================
        # 5. CREATE FOUR GRAPHS
        # ==================================

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))

        fig.suptitle(
            "Sales Performance Analysis",
            fontsize=17,
            fontweight="bold"
        )

        # Graph 1: Monthly revenue
        axes[0, 0].plot(
            monthly_revenue.index,
            monthly_revenue.values,
            marker="o"
        )
        axes[0, 0].set_title("Monthly Revenue")
        axes[0, 0].set_ylabel("Revenue (₹)")
        axes[0, 0].tick_params(axis="x", rotation=35)
        axes[0, 0].grid(True, alpha=0.3)

        # Graph 2: Revenue by category
        axes[0, 1].bar(
            category_revenue.index,
            category_revenue.values
        )
        axes[0, 1].set_title("Revenue by Category")
        axes[0, 1].set_ylabel("Revenue (₹)")
        axes[0, 1].tick_params(axis="x", rotation=25)

        # Graph 3: Revenue by region
        axes[1, 0].barh(
            region_revenue.index,
            region_revenue.values
        )
        axes[1, 0].set_title("Revenue by Region")
        axes[1, 0].set_xlabel("Revenue (₹)")

        # Graph 4: Monthly profit
        axes[1, 1].plot(
            monthly_profit.index,
            monthly_profit.values,
            marker="s"
        )
        axes[1, 1].set_title("Monthly Profit")
        axes[1, 1].set_ylabel("Profit (₹)")
        axes[1, 1].tick_params(axis="x", rotation=35)
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()
        plt.close(fig)

        # ==================================
        # 6. DISPLAY FILTERED DATA TABLE
        # ==================================

        print("\nFILTERED SALES DATA")

        display(
            filtered.sort_values(
                "Date", ascending=False
            ).reset_index(drop=True).style.format({
                "Unit_Price": "₹{:,.2f}",
                "Revenue": "₹{:,.2f}",
                "Cost": "₹{:,.2f}",
                "Profit": "₹{:,.2f}"
            })
        )

# ==========================================
# 7. CONNECT FILTERS TO DASHBOARD
# ==========================================

region_filter.observe(update_dashboard, names="value")
category_filter.observe(update_dashboard, names="value")

display(widgets.HTML(
    "<h2>Sales Data Science Dashboard</h2>"
    "<p>Analyze revenue, profit, products, and regions.</p>"
))

display(widgets.HBox([
    region_filter,
    category_filter
]))

display(output)

# ==========================================
# 8. CSV DOWNLOAD BUTTON
# ==========================================

download_output = widgets.Output()

download_button = widgets.Button(
    description="Export Filtered Data to CSV",
    button_style="success",
    icon="download"
)

def download_csv(button):
    filtered = data.copy()

    if region_filter.value != "All Regions":
        filtered = filtered[
            filtered["Region"] == region_filter.value
        ]

    if category_filter.value != "All Categories":
        filtered = filtered[
            filtered["Category"] == category_filter.value
        ]

    filename = "sales_dashboard_data.csv"
    filtered.to_csv(filename, index=False)

    with download_output:
        clear_output(wait=True)
        print("CSV exported successfully!")
        display(FileLink(filename))

download_button.on_click(download_csv)

display(download_button)
display(download_output)

# ==========================================
# 9. RUN THE DASHBOARD
# ==========================================

update_dashboard()

print("\nDashboard loaded successfully!")


In file included from <<< inputs >>>:1:
input_line_2:2:14: error: expected ';' after module name
    2 | import pandas as pd
      |              ^
      |              ;
input_line_2:2:8: fatal error: module 'pandas' not found
    2 | import pandas as pd
      | ~~~~~~~^~~~~~
Failed to parse via ::process:Parsing failed.


Error: : Compilation error! In file included from <<< inputs >>>:1:
input_line_2:2:14: error: expected ';' after module name
    2 | import pandas as pd
      |              ^
      |              ;
input_line_2:2:8: fatal error: module 'pandas' not found
    2 | import pandas as pd
      | ~~~~~~~^~~~~~
Failed to parse via ::process:Parsing failed.
